<a href="https://colab.research.google.com/github/alokchoudharyguliya/Human-Classifier/blob/main/Human_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Oct  7 07:49:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install kagglehub pyyaml pandas pillow numpy python-dotenv tensorflow==2.21.*

In [ ]:
!export KAGGLE_API_TOKEN=KGAT_7a14795b1a67fec360099ffe85c0e3b9

In [ ]:
# ============================================================
# HUMAN / NOT-HUMAN DATASET MANIFEST BUILDER
# Google Colab version
# ============================================================

# Install dependencies
!pip -q install kagglehub python-dotenv pillow


from __future__ import annotations

import argparse
import csv
import json
import os
import pickle
import random
import re
import sys
import tarfile
import urllib.request
from collections import defaultdict
from pathlib import Path


# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path("/content")

RAW_DIR = ROOT / "data" / "raw"
MANIFEST_DIR = ROOT / "data" / "manifests"
SPOTCHECK_DIR = ROOT / "data" / "spotcheck"

COMPETITION = "overnight-hackathon"

RULES_URL = (
    f"https://www.kaggle.com/competitions/{COMPETITION}/rules"
)

CIFAR_URL = (
    "https://www.cs.toronto.edu/~kriz/"
    "cifar-10-python.tar.gz"
)

CIFAR_ARCHIVE = RAW_DIR / "cifar-10-python.tar.gz"
CIFAR_BATCHES = RAW_DIR / "cifar-10-batches-py"
CIFAR_EXPORT = RAW_DIR / "cifar10"
CIFAR_SELECTION = CIFAR_EXPORT / "selection.json"

SKIP_DIRS = {
    "cifar10",
    "cifar-10-batches-py",
    "__MACOSX",
}

CIFAR_CLASSES = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}

GENDER_TOKENS = {
    "M": "M",
    "F": "F",
    "MALE": "M",
    "FEMALE": "F",
}

PLACEHOLDERS = {
    "",
    "your_token_here",
    "your_kaggle_username",
    "your_kaggle_key",
    "paste_here",
}

CSV_FIELDS = [
    "image_path",
    "split",
    "label",
    "age",
    "gender",
    "tag",
    "source_split",
]


# ============================================================
# MAIN
# ============================================================

def main(
    data_dir=RAW_DIR,
    skip_download=False,
    seed=42,
    output=MANIFEST_DIR / "labels.csv",
    spotcheck=20,
):

    data_dir = Path(data_dir).resolve()
    output = Path(output).resolve()

    print("=" * 60)
    print("HUMAN / NOT-HUMAN DATASET BUILDER")
    print("=" * 60)

    # --------------------------------------------------------
    # Download Overnight Hackathon dataset
    # --------------------------------------------------------

    if skip_download:
        dataset_root = data_dir

        if not dataset_root.exists():
            raise RuntimeError(
                f"No dataset found at {dataset_root}"
            )

        print(
            f"\nUsing existing files at:\n{dataset_root}"
        )

    else:
        load_credentials()
        dataset_root = download_competition(data_dir)

    # --------------------------------------------------------
    # Find human images
    # --------------------------------------------------------

    print("\nSearching for human images...")

    images = find_images(dataset_root)

    if not images:
        print("\nNo images found.")

        print("\nFirst levels of dataset:")
        print_tree(dataset_root)

        print(
            "\nIf Kaggle rejected the download, "
            "accept the competition rules here:"
        )

        print(RULES_URL)

        raise RuntimeError(
            "No human images found."
        )

    print(
        f"Found {len(images)} human images."
    )

    # --------------------------------------------------------
    # Build human rows
    # --------------------------------------------------------

    rows = [
        make_row(image, dataset_root)
        for image in images
    ]

    # --------------------------------------------------------
    # Split human data
    # --------------------------------------------------------

    print("\nCreating train/validation/test splits...")

    assign_splits(rows, seed)

    # --------------------------------------------------------
    # Add CIFAR-10
    # --------------------------------------------------------

    print("\nBuilding CIFAR-10 not-human dataset...")

    cifar_rows = build_cifar_rows(
        rows,
        seed
    )

    rows.extend(cifar_rows)

    # --------------------------------------------------------
    # Write outputs
    # --------------------------------------------------------

    write_outputs(
        rows,
        seed=seed,
        output=output,
        spotcheck=spotcheck,
    )

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print_summary(
        rows,
        dataset_root,
    )


# ============================================================
# KAGGLE AUTHENTICATION
# ============================================================

def load_credentials():

    print("\nChecking Kaggle credentials...")

    # --------------------------------------------------------
    # Try environment variables first
    # --------------------------------------------------------

    if credential_is_set("KAGGLE_API_TOKEN"):
        print("Using KAGGLE_API_TOKEN.")
        return

    if (
        credential_is_set("KAGGLE_USERNAME")
        and credential_is_set("KAGGLE_KEY")
    ):
        print(
            "Using KAGGLE_USERNAME and KAGGLE_KEY."
        )
        return

    # --------------------------------------------------------
    # Try Google Colab Secrets
    # --------------------------------------------------------

    if load_colab_secrets():
        return

    # --------------------------------------------------------
    # No credentials
    # --------------------------------------------------------

    raise RuntimeError(
        "\nKaggle credentials are missing.\n\n"
        "In Colab, add your Kaggle credentials as a Secret.\n\n"
        "Recommended:\n"
        "  Secret name: KAGGLE_API_TOKEN\n\n"
        "Get it from:\n"
        "https://www.kaggle.com/settings/api\n\n"
        "Also make sure you have accepted the competition rules:\n"
        f"{RULES_URL}"
    )


def credential_is_set(name):

    value = os.getenv(name, "")

    return value.strip() not in PLACEHOLDERS


def load_colab_secrets():

    try:
        from google.colab import userdata
    except ImportError:
        return False

    print("Checking Colab Secrets...")

    for name in (
        "KAGGLE_API_TOKEN",
        "KAGGLE_USERNAME",
        "KAGGLE_KEY",
    ):

        try:
            value = userdata.get(name)
        except Exception:
            continue

        if value:

            value = value.strip()

            if value and value not in PLACEHOLDERS:
                os.environ[name] = value

    if credential_is_set("KAGGLE_API_TOKEN"):

        print(
            "Using Colab secret: KAGGLE_API_TOKEN"
        )

        return True

    if (
        credential_is_set("KAGGLE_USERNAME")
        and credential_is_set("KAGGLE_KEY")
    ):

        print(
            "Using Colab secrets: "
            "KAGGLE_USERNAME + KAGGLE_KEY"
        )

        return True

    return False


# ============================================================
# DOWNLOAD KAGGLE COMPETITION
# ============================================================

def download_competition(output_dir):

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    import kagglehub

    print(
        f"\nDownloading Kaggle competition:\n"
        f"{COMPETITION}"
    )

    try:

        downloaded = kagglehub.competition_download(
            COMPETITION,
            output_dir=str(output_dir),
        )

    except Exception as error:

        message = str(error)

        print("\nKaggle download failed:")
        print(message)

        if (
            "403" in message
            or "401" in message
            or "rules" in message.lower()
            or "authenticated" in message.lower()
        ):

            print(
                "\nYou probably need to accept "
                "the competition rules:"
            )

            print(RULES_URL)

        raise

    path = Path(downloaded).resolve()

    print(
        f"\nCompetition dataset downloaded to:\n"
        f"{path}"
    )

    return path


# ============================================================
# FIND IMAGES
# ============================================================

def find_images(root):

    images = []

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        if any(
            part.startswith(".")
            or part in SKIP_DIRS
            for part in path.parts
        ):
            continue

        images.append(path)

    return sorted(images)


# ============================================================
# HUMAN ROW
# ============================================================

def make_row(image, root):

    parsed = parse_filename(
        image.stem
    )

    source = source_split(
        image,
        root
    )

    return {
        "image_path": rel(image),
        "split": source,
        "label": "human",
        "age": parsed["age"],
        "gender": parsed["gender"],
        "tag": parsed["tag"],
        "source_split": source,
    }


# ============================================================
# PARSE HUMAN FILENAME
# ============================================================

def parse_filename(stem):

    parts = [
        part
        for part in re.split(
            r"[_\-\s]+",
            stem
        )
        if part
    ]

    if (
        parts
        and parts[0].upper() == "IMG"
    ):
        parts = parts[1:]

    age = ""
    gender = ""
    tags = []

    if (
        parts
        and parts[0].isdigit()
    ):
        parts = parts[1:]

    for part in parts:

        upper = part.upper()

        if (
            upper in GENDER_TOKENS
            and not gender
        ):

            gender = GENDER_TOKENS[upper]

        elif (
            part.isdigit()
            and not age
        ):

            age = part

        else:

            tags.append(part)

    return {
        "age": age,
        "gender": gender,
        "tag": "_".join(tags),
    }


# ============================================================
# DETERMINE SOURCE SPLIT
# ============================================================

def source_split(path, root):

    try:

        parts = (
            path.resolve()
            .relative_to(root.resolve())
            .parts[:-1]
        )

    except ValueError:

        parts = path.parts[:-1]

    for part in parts:

        name = part.lower()

        if name.startswith("train"):
            return "train"

        if name.startswith("val"):
            return "val"

        if name.startswith("test"):
            return "test"

    return "unsplit"


# ============================================================
# CREATE HUMAN TRAIN / VAL / TEST
# ============================================================

def assign_splits(rows, seed):

    # Competition test images must remain unused
    for row in rows:

        if row["source_split"] == "test":
            row["split"] = "unused"

    pool = [
        row
        for row in rows
        if row["source_split"]
        in {"train", "unsplit", "val"}
    ]

    if not pool:

        print(
            "No training images found. "
            "Competition test images stay unused."
        )

        return

    has_val = any(
        row["source_split"] == "val"
        for row in pool
    )

    has_train = any(
        row["source_split"] == "train"
        for row in pool
    )

    if has_val and has_train:

        train_rows = [
            row
            for row in pool
            if row["source_split"] != "val"
        ]

        val_rows = [
            row
            for row in pool
            if row["source_split"] == "val"
        ]

        test_rows, train_rows = carve(
            train_rows,
            0.10,
            seed,
        )

    else:

        test_rows, rest = carve(
            pool,
            0.10,
            seed,
        )

        val_rows, train_rows = carve(
            rest,
            0.10 / 0.90,
            seed + 1,
        )

    apply_split(
        train_rows,
        "train"
    )

    apply_split(
        val_rows,
        "val"
    )

    apply_split(
        test_rows,
        "test"
    )


def carve(rows, fraction, seed):

    rng = random.Random(seed)

    group = sorted(
        rows,
        key=lambda row: row["image_path"]
    )

    rng.shuffle(group)

    count = len(group)

    take = int(
        round(count * fraction)
    )

    if count >= 2:

        take = min(
            max(take, 1),
            count - 1
        )

    else:

        take = 0

    return (
        group[:take],
        group[take:],
    )


def apply_split(rows, split):

    for row in rows:
        row["split"] = split


# ============================================================
# BUILD CIFAR-10 DATASET
# ============================================================

def build_cifar_rows(
    human_rows,
    seed,
):

    targets = {
        split: sum(
            row["label"] == "human"
            and row["split"] == split
            for row in human_rows
        )
        for split in (
            "train",
            "val",
            "test",
        )
    }

    print(
        "\nHuman split targets:"
    )

    for split, count in targets.items():
        print(
            f"  {split}: {count}"
        )

    if any(
        count == 0
        for count in targets.values()
    ):

        raise RuntimeError(
            "Need human images in "
            "train, val, and test."
        )

    # --------------------------------------------------------
    # Reuse previous CIFAR selection
    # --------------------------------------------------------

    cached = read_cifar_selection(
        seed,
        targets,
    )

    if cached is not None:

        print(
            "\nReusing existing CIFAR-10 sample."
        )

        return cached

    # --------------------------------------------------------
    # Download CIFAR
    # --------------------------------------------------------

    ensure_cifar_batches()

    train_pool = group_cifar_records(
        load_cifar_records("train")
    )

    test_pool = group_cifar_records(
        load_cifar_records("test")
    )

    rng = random.Random(seed)

    # --------------------------------------------------------
    # Stratified sampling
    # --------------------------------------------------------

    chosen = {

        "test": take_stratified(
            test_pool,
            targets["test"],
            rng,
        ),

        "train": take_stratified(
            train_pool,
            targets["train"],
            rng,
        ),

        "val": take_stratified(
            train_pool,
            targets["val"],
            rng,
        ),
    }

    # --------------------------------------------------------
    # Remove old generated PNGs
    # --------------------------------------------------------

    if CIFAR_EXPORT.exists():

        for path in CIFAR_EXPORT.rglob("*.png"):

            if path.is_file():
                path.unlink()

    # --------------------------------------------------------
    # Save CIFAR images
    # --------------------------------------------------------

    rows = []

    saved = 0

    total = sum(
        len(items)
        for items in chosen.values()
    )

    for split in (
        "train",
        "val",
        "test",
    ):

        for index, (
            class_id,
            pixels
        ) in enumerate(
            chosen[split]
        ):

            class_name = CIFAR_CLASSES[
                class_id
            ]

            folder = (
                CIFAR_EXPORT
                / split
                / class_name
            )

            folder.mkdir(
                parents=True,
                exist_ok=True
            )

            path = (
                folder
                / f"{index:05d}.png"
            )

            save_cifar_png(
                pixels,
                path
            )

            rows.append(
                cifar_row(
                    path,
                    split,
                    class_name,
                )
            )

            saved += 1

            if (
                saved % 5000 == 0
                or saved == total
            ):

                print(
                    f"Wrote {saved} / {total} "
                    "CIFAR-10 images"
                )

    # --------------------------------------------------------
    # Save selection metadata
    # --------------------------------------------------------

    CIFAR_EXPORT.mkdir(
        parents=True,
        exist_ok=True
    )

    selection = {
        "seed": seed,
        "targets": targets,
        "rows": rows,
    }

    CIFAR_SELECTION.write_text(
        json.dumps(
            selection,
            indent=2
        ) + "\n",
        encoding="utf-8",
    )

    return rows


# ============================================================
# CIFAR CACHE
# ============================================================

def read_cifar_selection(
    seed,
    targets,
):

    if not CIFAR_SELECTION.exists():
        return None

    try:

        payload = json.loads(
            CIFAR_SELECTION.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        return None

    if payload.get("seed") != seed:
        return None

    if payload.get("targets") != targets:
        return None

    rows = payload.get(
        "rows",
        []
    )

    if not rows:
        return None

    # Check generated files
    for row in (
        rows[0],
        rows[-1],
    ):

        if not (
            ROOT
            / row["image_path"]
        ).exists():

            return None

    return rows


# ============================================================
# DOWNLOAD CIFAR-10
# ============================================================

def ensure_cifar_batches():

    if (
        CIFAR_BATCHES / "data_batch_1"
    ).exists() and (
        CIFAR_BATCHES / "test_batch"
    ).exists():

        print(
            "Using existing CIFAR-10 batches."
        )

        return

    RAW_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Download
    # --------------------------------------------------------

    if not CIFAR_ARCHIVE.exists():

        print(
            "\nDownloading CIFAR-10..."
        )

        urllib.request.urlretrieve(
            CIFAR_URL,
            CIFAR_ARCHIVE,
            reporthook=download_progress,
        )

        print()

    # --------------------------------------------------------
    # Extract
    # --------------------------------------------------------

    print(
        "Extracting CIFAR-10..."
    )

    with tarfile.open(
        CIFAR_ARCHIVE
    ) as archive:

        archive.extractall(
            RAW_DIR
        )

    if not (
        CIFAR_BATCHES
        / "data_batch_1"
    ).exists():

        raise RuntimeError(
            "CIFAR-10 extraction failed."
        )


def download_progress(
    block_count,
    block_size,
    total_size,
):

    if total_size <= 0:
        return

    done = min(
        block_count * block_size,
        total_size
    )

    if (
        done == total_size
        or block_count % 200 == 0
    ):

        print(
            f"\rCIFAR-10 download: "
            f"{done / 1_048_576:.1f} / "
            f"{total_size / 1_048_576:.1f} MB",
            end="",
        )


# ============================================================
# LOAD CIFAR RECORDS
# ============================================================

def load_cifar_records(which):

    if which == "train":

        names = [
            f"data_batch_{index}"
            for index in range(1, 6)
        ]

    else:

        names = [
            "test_batch"
        ]

    records = []

    for name in names:

        with (
            CIFAR_BATCHES / name
        ).open("rb") as handle:

            batch = pickle.load(
                handle,
                encoding="bytes"
            )

        pixels = batch[b"data"]
        labels = batch[b"labels"]

        for index, label in enumerate(labels):

            records.append(
                (
                    int(label),
                    bytes(
                        pixels[index]
                    ),
                )
            )

    return records


# ============================================================
# GROUP CIFAR BY CLASS
# ============================================================

def group_cifar_records(records):

    grouped = defaultdict(list)

    for label, pixels in records:

        grouped[label].append(
            pixels
        )

    return grouped


# ============================================================
# STRATIFIED CIFAR SAMPLING
# ============================================================

def take_stratified(
    pools,
    count,
    rng,
):

    classes = sorted(pools)

    base, extra = divmod(
        count,
        len(classes)
    )

    chosen = []

    for index, class_id in enumerate(
        classes
    ):

        need = (
            base
            + (
                1
                if index < extra
                else 0
            )
        )

        pool = pools[class_id]

        rng.shuffle(pool)

        if need > len(pool):

            raise RuntimeError(
                f"Not enough CIFAR-10 "
                f"{CIFAR_CLASSES[class_id]} "
                f"images."
            )

        chosen.extend(
            (
                class_id,
                pixels
            )
            for pixels in pool[:need]
        )

        pools[class_id] = (
            pool[need:]
        )

    rng.shuffle(chosen)

    return chosen


# ============================================================
# SAVE CIFAR IMAGE
# ============================================================

def save_cifar_png(
    pixels,
    path,
):

    from PIL import Image

    red = pixels[
        0:1024
    ]

    green = pixels[
        1024:2048
    ]

    blue = pixels[
        2048:3072
    ]

    image = Image.merge(
        "RGB",
        (
            Image.frombytes(
                "L",
                (32, 32),
                red,
            ),

            Image.frombytes(
                "L",
                (32, 32),
                green,
            ),

            Image.frombytes(
                "L",
                (32, 32),
                blue,
            ),
        ),
    )

    image.save(path)


# ============================================================
# CIFAR CSV ROW
# ============================================================

def cifar_row(
    path,
    split,
    class_name,
):

    return {
        "image_path": rel(path),
        "split": split,
        "label": "not-human",
        "age": "",
        "gender": "",
        "tag": class_name,
        "source_split": "cifar10",
    }


# ============================================================
# WRITE OUTPUTS
# ============================================================

def write_outputs(
    rows,
    seed,
    output,
    spotcheck,
):

    output.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    ordered = sorted(
        rows,
        key=lambda row: (
            row["split"],
            row["image_path"],
        ),
    )

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    with output.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=CSV_FIELDS,
        )

        writer.writeheader()
        writer.writerows(ordered)

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    ages = sorted(
        {
            int(row["age"])
            for row in rows
            if row["age"].isdigit()
        }
    )

    usable = [
        row
        for row in rows
        if row["split"]
        in {
            "train",
            "val",
            "test",
        }
    ]

    meta = {
        "human_source": COMPETITION,
        "not_human_source": "cifar-10",
        "cifar_classes": list(
            CIFAR_CLASSES
        ),
        "task": (
            "human from Overnight Hackathon, "
            "not-human from CIFAR-10"
        ),
        "seed": seed,
        "age_min": (
            ages[0]
            if ages
            else None
        ),
        "age_max": (
            ages[-1]
            if ages
            else None
        ),
        "human_images": sum(
            row["label"] == "human"
            for row in usable
        ),
        "not_human_images": sum(
            row["label"] == "not-human"
            for row in usable
        ),
        "cifar_native_size": [
            32,
            32,
        ],
    }

    meta_path = (
        output.parent
        / "class_map.json"
    )

    meta_path.write_text(
        json.dumps(
            meta,
            indent=2
        ) + "\n",
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Spot check
    # --------------------------------------------------------

    write_spotcheck(
        ordered,
        spotcheck,
        seed,
    )

    print(
        f"\nWrote:\n{output}"
    )

    print(
        f"Wrote:\n{meta_path}"
    )


# ============================================================
# SPOT CHECK
# ============================================================

def write_spotcheck(
    rows,
    count,
    seed,
):

    if count <= 0:
        return

    if SPOTCHECK_DIR.exists():

        for child in (
            SPOTCHECK_DIR.rglob("*")
        ):

            if (
                child.is_symlink()
                or child.is_file()
            ):

                child.unlink()

    rng = random.Random(seed)

    MANIFEST_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    for label in (
        "human",
        "not-human",
    ):

        folder = (
            SPOTCHECK_DIR
            / label
        )

        folder.mkdir(
            parents=True,
            exist_ok=True
        )

        pool = sorted(
            (
                row
                for row in rows
                if row["split"] == "train"
                and row["label"] == label
            ),
            key=lambda row: row[
                "image_path"
            ],
        )

        rng.shuffle(pool)

        lines = []

        for index, row in enumerate(
            pool[:count],
            start=1,
        ):

            source = (
                ROOT
                / row["image_path"]
            ).resolve()

            link = (
                folder
                / f"{index:02d}_{source.name}"
            )

            if (
                link.exists()
                or link.is_symlink()
            ):

                link.unlink()

            link.symlink_to(
                source
            )

            lines.append(
                row["image_path"]
            )

        if label == "human":

            list_name = (
                "spotcheck_human.txt"
            )

        else:

            list_name = (
                "spotcheck_not_human.txt"
            )

        (
            MANIFEST_DIR
            / list_name
        ).write_text(
            "\n".join(lines)
            + (
                "\n"
                if lines
                else ""
            ),
            encoding="utf-8",
        )


# ============================================================
# SUMMARY
# ============================================================

def print_summary(
    rows,
    dataset_root,
):

    human_rows = [
        row
        for row in rows
        if row["label"] == "human"
    ]

    cifar_rows = [
        row
        for row in rows
        if row["label"] == "not-human"
    ]

    print("\n")
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)

    print(
        f"\nDataset root:\n{dataset_root}"
    )

    print(
        "\nHuman: Overnight Hackathon person photos."
    )

    print(
        "Not-human: CIFAR-10 objects and animals."
    )

    print(
        "CIFAR-10 native resolution: 32x32."
    )

    print(
        "\nTotal images:",
        len(rows)
    )

    print(
        "Human:",
        len(human_rows)
    )

    print(
        "Not-human:",
        len(cifar_rows)
    )

    print("\nSplit counts:")

    for split in (
        "train",
        "val",
        "test",
        "unused",
    ):

        group = [
            row
            for row in rows
            if row["split"] == split
        ]

        human_count = sum(
            row["label"] == "human"
            for row in group
        )

        other_count = sum(
            row["label"] == "not-human"
            for row in group
        )

        print(
            f"  {split}: "
            f"{len(group)} "
            f"(human {human_count}, "
            f"not-human {other_count})"
        )

    print(
        "\nCIFAR-10 classes in train:"
    )

    train_tags = defaultdict(int)

    for row in cifar_rows:

        if row["split"] == "train":

            train_tags[
                row["tag"]
            ] += 1

    for name, count in sorted(
        train_tags.items()
    ):

        print(
            f"  {name}: {count}"
        )

    print(
        "\nOutput directory:"
    )

    print(
        MANIFEST_DIR
    )

    print(
        "\nSpot-check directories:"
    )

    print(
        SPOTCHECK_DIR / "human"
    )

    print(
        SPOTCHECK_DIR / "not-human"
    )

    print("\nDone.")


# ============================================================
# PRINT DIRECTORY TREE
# ============================================================

def print_tree(
    root,
    max_depth=2,
):

    root = root.resolve()

    print(root)

    for path in sorted(
        root.rglob("*")
    ):

        try:

            depth = len(
                path.relative_to(
                    root
                ).parts
            )

        except ValueError:

            continue

        if depth > max_depth:
            continue

        suffix = (
            "/"
            if path.is_dir()
            else ""
        )

        print(
            "  " * depth
            + path.name
            + suffix
        )


# ============================================================
# RELATIVE PATH
# ============================================================

def rel(path):

    resolved = path.resolve()

    try:

        return str(
            resolved.relative_to(
                ROOT
            )
        )

    except ValueError:

        return str(resolved)


# ============================================================
# RUN
# ============================================================

main(
    data_dir=RAW_DIR,
    skip_download=False,
    seed=42,
    output=MANIFEST_DIR / "labels.csv",
    spotcheck=20,
)

HUMAN / NOT-HUMAN DATASET BUILDER

Checking Kaggle credentials...
Checking Colab Secrets...
Using Colab secret: KAGGLE_API_TOKEN

overnight-hackathon


100%|██████████| 337M/337M [00:03<00:00, 115MB/s]

Extracting files...



Competition dataset downloaded to:
/content/data/raw

Searching for human images...
Found 42139 human images.

Creating train/validation/test splits...

Building CIFAR-10 not-human dataset...

Human split targets:
  train: 23597
  val: 2950
  test: 2950

CIFAR-10 download: 162.6 / 162.6 MB
Extracting CIFAR-10...


/tmp/ipykernel_2114/3254251777.py:978: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(


Wrote 5000 / 29497 CIFAR-10 images
Wrote 10000 / 29497 CIFAR-10 images
Wrote 15000 / 29497 CIFAR-10 images
Wrote 20000 / 29497 CIFAR-10 images
Wrote 25000 / 29497 CIFAR-10 images
Wrote 29497 / 29497 CIFAR-10 images

Wrote:
/content/data/manifests/labels.csv
Wrote:
/content/data/manifests/class_map.json


DATASET SUMMARY

Dataset root:
/content/data/raw

Human: Overnight Hackathon person photos.
Not-human: CIFAR-10 objects and animals.
CIFAR-10 native resolution: 32x32.

Total images: 71636
Human: 42139
Not-human: 29497

Split counts:
  train: 47194 (human 23597, not-human 23597)
  val: 5900 (human 2950, not-human 2950)
  test: 5900 (human 2950, not-human 2950)
  unused: 12642 (human 12642, not-human 0)

CIFAR-10 classes in train:
  airplane: 2360
  automobile: 2360
  bird: 2360
  cat: 2360
  deer: 2360
  dog: 2360
  frog: 2360
  horse: 2359
  ship: 2359
  truck: 2359

Output directory:
/content/data/manifests

Spot-check directories:
/content/data/spotcheck/human
/content/data/spotchec

In [ ]:
# ============================================================
# PHASE 2
# Train Human / Not-Human Classifier
# Google Colab version
# ============================================================

# Install dependencies
!pip -q install tensorflow pillow


# ============================================================
# IMPORTS
# ============================================================

from __future__ import annotations

import csv
import json
import random
import sys
from pathlib import Path


import tensorflow as tf


# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path("/content")

MANIFEST = (
    ROOT
    / "data"
    / "manifests"
    / "labels.csv"
)

NOT_HUMAN_DIR = (
    ROOT
    / "data"
    / "not_human"
)

MODEL_DIR = (
    ROOT
    / "models"
)

IMG_SIZE = 224
BATCH_SIZE = 32

HEAD_EPOCHS = 5
FINE_EPOCHS = 15

SEED = 42

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 60)
    print("PHASE 2: HUMAN / NOT-HUMAN CLASSIFIER")
    print("=" * 60)

    # --------------------------------------------------------
    # Check TensorFlow / GPU
    # --------------------------------------------------------

    report_device(tf)

    tf.keras.utils.set_random_seed(SEED)

    # --------------------------------------------------------
    # Load Phase 1 manifest
    # --------------------------------------------------------

    rows = load_manifest()

    print(
        f"\nLoaded {len(rows)} rows from:"
    )

    print(MANIFEST)

    # --------------------------------------------------------
    # Add additional not-human images if present
    # --------------------------------------------------------

    rows = add_not_human(rows)

    # --------------------------------------------------------
    # Make sure both classes exist
    # --------------------------------------------------------

    require_both_classes(rows)

    # --------------------------------------------------------
    # Save final training split
    # --------------------------------------------------------

    write_training_split(rows)

    # --------------------------------------------------------
    # Create TensorFlow datasets
    # --------------------------------------------------------

    print("\nCreating training dataset...")

    train_ds = make_dataset(
        tf,
        rows,
        "train",
        training=True,
    )

    print("Creating validation dataset...")

    val_ds = make_dataset(
        tf,
        rows,
        "val",
        training=False,
    )

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    model = build_model(tf)

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------

    class_weight = make_class_weight(rows)

    print(
        "\nClass weights:"
    )

    print(class_weight)

    # --------------------------------------------------------
    # Model directory
    # --------------------------------------------------------

    MODEL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        MODEL_DIR / "best_model.keras",
        monitor="val_loss",
        save_best_only=True,
    )

    callbacks = make_callbacks(
        tf,
        checkpoint,
    )

    # ========================================================
    # STAGE 1
    # Train classification head
    # ========================================================

    print("\n")
    print("=" * 60)
    print("STAGE 1: TRAINING CLASSIFICATION HEAD")
    print("=" * 60)

    print(
        "\nBackbone is frozen."
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    # ========================================================
    # STAGE 2
    # Fine-tune top MobileNetV2 layers
    # ========================================================

    print("\n")
    print("=" * 60)
    print("STAGE 2: FINE-TUNING")
    print("=" * 60)

    unfreeze_top(model)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4
        ),
        loss="binary_crossentropy",
        metrics=metrics(tf),
    )

    finished = len(
        history.history["loss"]
    )

    print(
        f"\nFine-tuning for {FINE_EPOCHS} additional epochs."
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=finished + FINE_EPOCHS,
        initial_epoch=finished,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    best = (
        MODEL_DIR
        / "best_model.keras"
    )

    if not best.exists():

        raise RuntimeError(
            "best_model.keras was not created."
        )

    print(
        "\nLoading best checkpoint..."
    )

    model = tf.keras.models.load_model(
        best
    )

    # --------------------------------------------------------
    # Save final model
    # --------------------------------------------------------

    final_model = (
        MODEL_DIR
        / "human_classifier.keras"
    )

    model.save(
        final_model
    )

    # --------------------------------------------------------
    # Save preprocessing information
    # --------------------------------------------------------

    write_preprocess_spec()

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    print("\n")
    print("=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)

    print(
        f"\nBest model:\n{best}"
    )

    print(
        f"\nFinal model:\n{final_model}"
    )

    print(
        f"\nPreprocessing spec:\n"
        f"{MODEL_DIR / 'preprocess.json'}"
    )


# ============================================================
# GPU / DEVICE
# ============================================================

def report_device(tf):

    print("\nTensorFlow version:")
    print(tf.__version__)

    gpus = tf.config.list_physical_devices(
        "GPU"
    )

    if not gpus:

        print(
            "\nWARNING: No GPU visible to TensorFlow."
        )

        print(
            "Training will use CPU."
        )

        print(
            "\nIn Colab enable:"
        )

        print(
            "Runtime -> Change runtime type -> GPU"
        )

        return

    print(
        "\nGPU detected:"
    )

    for gpu in gpus:

        try:

            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

        except Exception:

            pass

        print(
            f"  {gpu.name}"
        )


# ============================================================
# LOAD MANIFEST
# ============================================================

def load_manifest():

    if not MANIFEST.exists():

        raise FileNotFoundError(
            f"\nMissing manifest:\n{MANIFEST}\n\n"
            "Run Phase 1 first."
        )

    with MANIFEST.open(
        newline="",
        encoding="utf-8",
    ) as handle:

        return list(
            csv.DictReader(handle)
        )


# ============================================================
# ADD OPTIONAL NOT-HUMAN DATA
# ============================================================

def add_not_human(rows):

    images = find_images(
        NOT_HUMAN_DIR
    )

    # No additional directory
    if not images:

        print(
            "\nNo additional images found in:"
        )

        print(
            NOT_HUMAN_DIR
        )

        return rows

    known = {
        row["image_path"]
        for row in rows
    }

    fresh = [
        path
        for path in images
        if rel(path) not in known
    ]

    if not fresh:

        return rows

    grouped = group_by_split_dir(
        fresh
    )

    if grouped:

        extra = []

        for split, paths in grouped.items():

            extra.extend(
                not_human_row(
                    path,
                    split
                )
                for path in paths
            )

    else:

        extra = assign_not_human_splits(
            fresh,
            SEED,
        )

    print(
        f"\nAdded {len(extra)} "
        f"not-human images from:"
    )

    print(
        NOT_HUMAN_DIR
    )

    return rows + extra


# ============================================================
# REQUIRE BOTH CLASSES
# ============================================================

def require_both_classes(rows):

    usable = [
        row
        for row in rows
        if row["split"]
        in {
            "train",
            "val",
            "test",
        }
    ]

    labels = {
        row["label"]
        for row in usable
    }

    if (
        "human" not in labels
        or "not-human" not in labels
    ):

        human_count = sum(
            row["label"] == "human"
            for row in usable
        )

        not_human_count = sum(
            row["label"] == "not-human"
            for row in usable
        )

        raise RuntimeError(
            "\nTraining needs BOTH classes.\n\n"
            f"Human images: {human_count}\n"
            f"Not-human images: {not_human_count}\n\n"
            "A one-class model would simply learn "
            "to predict human for everything."
        )

    train_counts = {
        label: sum(
            row["label"] == label
            and row["split"] == "train"
            for row in rows
        )
        for label in (
            "human",
            "not-human",
        )
    }

    if (
        train_counts["human"] == 0
        or train_counts["not-human"] == 0
    ):

        raise RuntimeError(
            "The train split needs both classes.\n"
            f"Current counts: {train_counts}"
        )

    print(
        "\nTraining class counts:"
    )

    print(
        f"  human:     {train_counts['human']}"
    )

    print(
        f"  not-human: {train_counts['not-human']}"
    )


# ============================================================
# FIND IMAGES
# ============================================================

def find_images(folder):

    if not folder.exists():

        return []

    images = []

    for path in folder.rglob("*"):

        if not path.is_file():
            continue

        if (
            path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        if any(
            part.startswith(".")
            for part in path.parts
        ):

            continue

        images.append(path)

    return sorted(images)


# ============================================================
# DETECT PREDEFINED SPLIT DIRECTORIES
# ============================================================

def group_by_split_dir(paths):

    grouped = {
        "train": [],
        "val": [],
        "test": [],
    }

    saw_split_dir = False

    for path in paths:

        try:

            parts = (
                path.resolve()
                .relative_to(
                    NOT_HUMAN_DIR.resolve()
                )
                .parts[:-1]
            )

        except ValueError:

            parts = path.parts[:-1]

        split = ""

        for part in parts:

            name = part.lower()

            if name.startswith("train"):

                split = "train"

            elif name.startswith("val"):

                split = "val"

            elif name.startswith("test"):

                split = "test"

        if split:

            saw_split_dir = True

            grouped[
                split
            ].append(path)

    if not saw_split_dir:

        return {}

    return grouped


# ============================================================
# SPLIT ADDITIONAL NOT-HUMAN IMAGES
# ============================================================

def assign_not_human_splits(
    paths,
    seed,
):

    ordered = sorted(
        paths,
        key=lambda path: rel(path)
    )

    random.Random(seed).shuffle(
        ordered
    )

    count = len(ordered)

    # 10% test
    test_count = (
        min(
            max(
                int(round(count * 0.10)),
                1,
            ),
            count - 1,
        )
        if count >= 2
        else 0
    )

    rest = ordered[
        test_count:
    ]

    # Approximately 10% validation
    val_count = (
        min(
            max(
                int(
                    round(
                        len(rest)
                        * (0.10 / 0.90)
                    )
                ),
                1,
            ),
            len(rest) - 1,
        )
        if len(rest) >= 2
        else 0
    )

    rows = []

    for path in ordered[
        :test_count
    ]:

        rows.append(
            not_human_row(
                path,
                "test",
            )
        )

    for path in rest[
        :val_count
    ]:

        rows.append(
            not_human_row(
                path,
                "val",
            )
        )

    for path in rest[
        val_count:
    ]:

        rows.append(
            not_human_row(
                path,
                "train",
            )
        )

    return rows


# ============================================================
# CREATE NOT-HUMAN ROW
# ============================================================

def not_human_row(
    path,
    split,
):

    return {
        "image_path": rel(path),
        "split": split,
        "label": "not-human",
        "age": "",
        "gender": "",
        "tag": "",
        "source_split": "not_human",
    }


# ============================================================
# WRITE TRAINING SPLIT
# ============================================================

def write_training_split(rows):

    usable = [
        row
        for row in rows
        if row["split"]
        in {
            "train",
            "val",
            "test",
        }
    ]

    path = (
        ROOT
        / "data"
        / "manifests"
        / "training_split.csv"
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fields = [
        "image_path",
        "split",
        "label",
        "age",
        "gender",
        "tag",
        "source_split",
    ]

    with path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            extrasaction="ignore",
        )

        writer.writeheader()

        writer.writerows(
            sorted(
                usable,
                key=lambda row: (
                    row["split"],
                    row["label"],
                    row["image_path"],
                ),
            )
        )

    print(
        f"\nWrote training split:"
    )

    print(path)


# ============================================================
# CREATE TF.DATA DATASET
# ============================================================

def make_dataset(
    tf,
    rows,
    split,
    training,
):

    chosen = [
        row
        for row in rows
        if row["split"] == split
    ]

    if not chosen:

        raise RuntimeError(
            f"No images in {split} split."
        )

    paths = [
        str(
            (
                ROOT
                / row["image_path"]
            ).resolve()
        )
        for row in chosen
    ]

    labels = [
        1
        if row["label"] == "human"
        else 0
        for row in chosen
    ]

    print(
        f"{split}: {len(paths)} images"
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (
            paths,
            labels,
        )
    )

    if training:

        ds = ds.shuffle(
            len(paths),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

        ds = ds.map(
            lambda path, label:
                augment_example(
                    tf,
                    path,
                    label,
                ),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    else:

        ds = ds.map(
            lambda path, label:
                (
                    load_image(
                        tf,
                        path,
                    ),
                    label,
                ),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    # Ignore corrupt images rather than
    # crashing the complete training run.
    ds = ds.ignore_errors()

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


# ============================================================
# IMAGE LOADING
# ============================================================

def load_image(
    tf,
    path,
):

    lower = tf.strings.lower(
        path
    )

    extension = tf.strings.substr(
        lower,
        tf.strings.length(lower) - 4,
        4,
    )

    is_bmp = tf.equal(
        extension,
        ".bmp",
    )

    image = tf.cond(
        is_bmp,

        lambda:
            decode_pillow(
                tf,
                path,
            ),

        lambda:
            decode_tensorflow(
                tf,
                path,
            ),
    )

    image.set_shape(
        [
            IMG_SIZE,
            IMG_SIZE,
            3,
        ]
    )

    return image


# ============================================================
# NORMAL IMAGE DECODER
# ============================================================

def decode_tensorflow(
    tf,
    path,
):

    image = tf.io.read_file(
        path
    )

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False,
    )

    image.set_shape(
        [
            None,
            None,
            3,
        ]
    )

    image = tf.image.resize(
        image,
        [
            IMG_SIZE,
            IMG_SIZE,
        ],
    )

    image.set_shape(
        [
            IMG_SIZE,
            IMG_SIZE,
            3,
        ]
    )

    return image


# ============================================================
# BMP DECODER
# ============================================================

def decode_pillow(
    tf,
    path,
):

    """
    PIL decoder for problematic BMP files.
    """

    def _decode(
        path_tensor
    ):

        import numpy as np
        from PIL import Image

        file_path = (
            path_tensor
            .numpy()
            .decode("utf-8")
        )

        with Image.open(
            file_path
        ) as image:

            image = (
                image
                .convert("RGB")
                .resize(
                    (
                        IMG_SIZE,
                        IMG_SIZE,
                    ),
                    Image.BILINEAR,
                )
            )

            return np.asarray(
                image,
                dtype=np.float32,
            )

    image = tf.py_function(
        _decode,
        [path],
        tf.float32,
    )

    image.set_shape(
        [
            IMG_SIZE,
            IMG_SIZE,
            3,
        ]
    )

    return image


# ============================================================
# DATA AUGMENTATION
# ============================================================

def augment_example(
    tf,
    path,
    label,
):

    """
    Augmentation happens on CPU.
    """

    with tf.device(
        "/CPU:0"
    ):

        image = load_image(
            tf,
            path,
        )

        # Horizontal flip
        image = tf.image.random_flip_left_right(
            image
        )

        # Brightness
        image = tf.image.random_brightness(
            image,
            max_delta=38.0,
        )

        # Contrast
        image = tf.image.random_contrast(
            image,
            lower=0.9,
            upper=1.1,
        )

        # Random zoom
        image = random_zoom(
            tf,
            image,
        )

        image = tf.clip_by_value(
            image,
            0.0,
            255.0,
        )

    return image, label


# ============================================================
# RANDOM ZOOM
# ============================================================

def random_zoom(
    tf,
    image,
):

    margin = tf.random.uniform(
        [],
        0,
        12,
        dtype=tf.int32,
    )

    crop = (
        IMG_SIZE
        - (2 * margin)
    )

    image = tf.slice(
        image,
        tf.stack(
            [
                margin,
                margin,
                0,
            ]
        ),
        tf.stack(
            [
                crop,
                crop,
                3,
            ]
        ),
    )

    image = tf.image.resize(
        image,
        [
            IMG_SIZE,
            IMG_SIZE,
        ],
    )

    image.set_shape(
        [
            IMG_SIZE,
            IMG_SIZE,
            3,
        ]
    )

    return image


# ============================================================
# BUILD MOBILENETV2 MODEL
# ============================================================

def build_model(tf):

    inputs = tf.keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        )
    )

    # Convert [0,255] -> [-1,1]
    x = tf.keras.layers.Rescaling(
        scale=1.0 / 127.5,
        offset=-1.0,
    )(inputs)

    base = (
        tf.keras.applications.MobileNetV2(
            input_shape=(
                IMG_SIZE,
                IMG_SIZE,
                3,
            ),
            include_top=False,
            weights="imagenet",
            alpha=0.75,
        )
    )

    # Freeze backbone for stage 1
    base.trainable = False

    x = base(
        x,
        training=False,
    )

    x = (
        tf.keras.layers.GlobalAveragePooling2D()
        (x)
    )

    x = (
        tf.keras.layers.Dropout(0.2)
        (x)
    )

    outputs = (
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="human_probability",
        )
        (x)
    )

    model = tf.keras.Model(
        inputs,
        outputs,
        name="human_classifier",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="binary_crossentropy",
        metrics=metrics(tf),
    )

    model.summary()

    return model


# ============================================================
# UNFREEZE TOP BACKBONE LAYERS
# ============================================================

def unfreeze_top(model):

    base = next(
        (
            layer
            for layer in model.layers
            if layer.name.startswith(
                "mobilenetv2"
            )
        ),
        None,
    )

    if base is None:

        raise RuntimeError(
            "Could not find MobileNetV2 backbone."
        )

    base.trainable = True

    frozen = max(
        len(base.layers) - 40,
        0,
    )

    for layer in base.layers[
        :frozen
    ]:

        layer.trainable = False

    print(
        f"\nUnfroze "
        f"{len(base.layers) - frozen} "
        "MobileNetV2 layers."
    )


# ============================================================
# METRICS
# ============================================================

def metrics(tf):

    return [
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),
    ]


# ============================================================
# CLASS WEIGHTS
# ============================================================

def make_class_weight(rows):

    train_rows = [
        row
        for row in rows
        if row["split"] == "train"
    ]

    negatives = sum(
        row["label"] == "not-human"
        for row in train_rows
    )

    positives = sum(
        row["label"] == "human"
        for row in train_rows
    )

    if negatives == 0 or positives == 0:

        raise RuntimeError(
            "Cannot calculate class weights "
            "because one class is missing."
        )

    total = (
        negatives
        + positives
    )

    return {
        0: total / (
            2 * negatives
        ),

        1: total / (
            2 * positives
        ),
    }


# ============================================================
# CALLBACKS
# ============================================================

def make_callbacks(
    tf,
    checkpoint,
):

    return [

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
        ),

        checkpoint,

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
        ),
    ]


# ============================================================
# PREPROCESSING SPEC
# ============================================================

def write_preprocess_spec():

    spec = {

        "input_size": IMG_SIZE,

        "color": "rgb",

        "input_range": "0-255",

        "normalization": (
            "included_in_model"
        ),

        "positive_class": "human",

        "positive_value": 1,

        "negative_class": "not-human",

        "negative_value": 0,

        "threshold": 0.5,
    }

    path = (
        MODEL_DIR
        / "preprocess.json"
    )

    path.write_text(
        json.dumps(
            spec,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    print(
        f"\nWrote:\n{path}"
    )


# ============================================================
# RELATIVE PATH
# ============================================================

def rel(path):

    resolved = path.resolve()

    try:

        return str(
            resolved.relative_to(
                ROOT
            )
        )

    except ValueError:

        return str(
            resolved
        )


# ============================================================
# RUN
# ============================================================

main()

PHASE 2: HUMAN / NOT-HUMAN CLASSIFIER

TensorFlow version:
2.21.0

GPU detected:
  /physical_device:GPU:0

Loaded 71636 rows from:
/content/data/manifests/labels.csv

No additional images found in:
/content/data/not_human

Training class counts:
  human:     23597
  not-human: 23597

Wrote training split:
/content/data/manifests/training_split.csv

Creating training dataset...
train: 47194 images
Creating validation dataset...
val: 5900 images
5903360/5903360 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "human_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_0.75_224            │ (None, 7, 7, 1280)     │     1,382,064 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ human_probability (Dense)       │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,383,345 (5.28 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 1,382,064 (5.27 MB)


Class weights:
{0: 1.0, 1: 1.0}


STAGE 1: TRAINING CLASSIFICATION HEAD

Backbone is frozen.
Epoch 1/5
   1475/Unknown 101s 60ms/step - accuracy: 0.9644 - loss: 0.0957 - precision: 0.9736 - recall: 0.9527

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1475/1475 ━━━━━━━━━━━━━━━━━━━━ 117s 71ms/step - accuracy: 0.9863 - loss: 0.0428 - precision: 0.9908 - recall: 0.9817 - val_accuracy: 0.9973 - val_loss: 0.0109 - val_precision: 1.0000 - val_recall: 0.9946 - learning_rate: 0.0010
Epoch 2/5
1475/1475 ━━━━━━━━━━━━━━━━━━━━ 88s 60ms/step - accuracy: 0.9954 - loss: 0.0150 - precision: 0.9968 - recall: 0.9940 - val_accuracy: 0.9975 - val_loss: 0.0073 - val_precision: 0.9973 - val_recall: 0.9976 - learning_rate: 0.0010
Epoch 3/5
1475/1475 ━━━━━━━━━━━━━━━━━━━━ 89s 60ms/step - accuracy: 0.9961 - loss: 0.0121 - precision: 0.9971 - recall: 0.9951 - val_accuracy: 0.9985 - val_loss: 0.0058 - val_precision: 1.0000 - val_recall: 0.9969 - learning_rate: 0.0010
Epoch 4/5
1475/1475 ━━━━━━━━━━━━━━━━━━━━ 144s 61ms/step - accuracy: 0.9963 - loss: 0.0108 - precision: 0.9972 - recall: 0.9953 - val_accuracy: 0.9986 - val_loss: 0.0047 - val_precision: 0.9997 - val_recall: 0.9976 - learning_rate: 0.0010
Epoch 5/5
1475/1475 ━━━━━━━━━━━━━━━━━━━━ 141s 96ms/step - ac

In [ ]:
import shutil

# This creates a file named models.zip from your 'models' folder
shutil.make_archive('models', 'zip', 'models')
print('Folder zipped successfully!')

Folder zipped successfully!
